# 01 — Build the spurious-text dataset

Reproduces the synthetic dataset from Section 7 of MILAN (Hernandez et al., ICLR 2022).
We start from Imagenette (10 ImageNet classes), paint the *correct* class name into the corner of half of the training images, and paint a *wrong* class name into every test image. The trained ResNet18 will learn the painted-text shortcut on training data, then fail on the adversarial test set — which is precisely what we'll fix by ablating MILAN-identified text neurons.

In [ ]:
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd().parent / 'milan'))
os.environ.setdefault('MILAN_DATA_DIR', str(Path.cwd().parent / 'data'))
os.environ.setdefault('MILAN_MODELS_DIR', str(Path.cwd().parent / 'models'))
os.environ.setdefault('MILAN_RESULTS_DIR', str(Path.cwd().parent / 'results'))

In [ ]:
from milan_repro.data.build_splits import download_imagenette, build
from milan_repro.data.render_text import TextStyle
import yaml

cfg = yaml.safe_load(open('../configs/resnet18_appendixE.yaml'))
style = TextStyle(
    font_size=cfg['text_overlay']['font_size'],
    font_color=tuple(cfg['text_overlay']['font_color']),
    stripe_color=tuple(cfg['text_overlay']['stripe_color']),
    stripe_alpha=cfg['text_overlay']['stripe_alpha'],
    margin_px=cfg['text_overlay']['margin_px'],
)
data_dir = Path(os.environ['MILAN_DATA_DIR'])
imagenette = download_imagenette(data_dir / 'imagenette')
version_dir = build(imagenette, data_dir / 'imagenet-spurious-text',
                    version='50pct',
                    train_fraction=cfg['text_overlay']['train_fraction'],
                    test_fraction=cfg['text_overlay']['test_fraction'],
                    style=style, seed=0)
print('dataset at', version_dir)

## Sanity-check: show some samples
Expect: training images sometimes have a correct overlay; test images always have a *wrong* overlay.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random
rng = random.Random(7)
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for col in range(8):
    for row, split in enumerate(['train', 'test']):
        cls = rng.choice(list((version_dir / split).iterdir()))
        img = rng.choice(list(cls.iterdir()))
        axes[row, col].imshow(Image.open(img))
        axes[row, col].set_axis_off()
        axes[row, col].set_title(f'{split}/{cls.name}', fontsize=7)
fig.tight_layout()